# Notebook 01 – Exploratory Data Analysis

- Load and inspect rating.csv + movie.csv
- Rating distribution
- User and item activity distributions
- Sparsity analysis
- Create and save train/test split


In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from src.preprocessing import (
    load_ratings, load_movies, encode_ids,
    per_user_split, sparsity_report
)

sns.set_style('whitegrid')
os.makedirs('../results', exist_ok=True)
DATA_DIR = '../data'
SEED     = 42

## 1.1 Load data

In [ ]:
# Use sample_frac=0.1 for quick development; remove for full run
ratings = load_ratings(DATA_DIR, sample_frac=None, seed=SEED)
movies  = load_movies(DATA_DIR)
print(f'Ratings shape: {ratings.shape}')
print(f'Movies  shape: {movies.shape}')
ratings.head()

In [ ]:
print('Dtypes:')
print(ratings.dtypes)
print('\nMissing values:')
print(ratings.isnull().sum())
print('\nBasic stats:')
print(ratings['rating'].describe())

## 1.2 Rating distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

ratings['rating'].hist(bins=9, ax=axes[0], color='steelblue', edgecolor='white')
axes[0].set_title('Rating Distribution')
axes[0].set_xlabel('Rating')
axes[0].set_ylabel('Count')

ratings['rating'].value_counts().sort_index().plot(
    kind='bar', ax=axes[1], color='steelblue', edgecolor='white')
axes[1].set_title('Rating Value Counts')
axes[1].set_xlabel('Rating')
axes[1].set_ylabel('Count')
axes[1].tick_params(axis='x', rotation=0)

plt.tight_layout()
plt.savefig('../results/rating_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

## 1.3 User & item activity (power-law)

In [ ]:
ratings_per_user  = ratings.groupby('userId').size()
ratings_per_movie = ratings.groupby('movieId').size()

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

axes[0].hist(ratings_per_user,  bins=50, log=True, color='coral', edgecolor='white')
axes[0].set_title('Ratings per User (log y)')
axes[0].set_xlabel('# ratings')
axes[0].set_ylabel('# users (log scale)')

axes[1].hist(ratings_per_movie, bins=50, log=True, color='seagreen', edgecolor='white')
axes[1].set_title('Ratings per Movie (log y)')
axes[1].set_xlabel('# ratings')
axes[1].set_ylabel('# movies (log scale)')

plt.tight_layout()
plt.savefig('../results/activity_distributions.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'Avg ratings per user : {ratings_per_user.mean():.1f}')
print(f'Med ratings per user : {ratings_per_user.median():.1f}')
print(f'Avg ratings per movie: {ratings_per_movie.mean():.1f}')
print(f'Med ratings per movie: {ratings_per_movie.median():.1f}')

## 1.4 Encode IDs + sparsity report

In [ ]:
ratings_enc, user_enc, movie_enc = encode_ids(ratings)
sparsity_report(ratings_enc)
print('New columns added:', [c for c in ratings_enc.columns if c not in ratings.columns])

## 1.5 Train / test split

In [ ]:
train, test = per_user_split(ratings_enc, test_ratio=0.2, seed=SEED)

print(f'Train size : {len(train):,}')
print(f'Test  size : {len(test):,}')
print(f'Test  ratio: {len(test)/len(ratings_enc):.1%}')
print(f'Users in test : {test["user_idx"].nunique():,} / {ratings_enc["user_idx"].nunique():,}')

# Save splits for other notebooks
train.to_parquet('../data/train.parquet', index=False)
test.to_parquet('../data/test.parquet',   index=False)
print('\nSplits saved to data/')

## 1.6 Genre distribution

In [ ]:
genres = movies['genres'].str.split('|').explode()
genre_counts = genres[genres != '(no genres listed)'].value_counts().head(15)

genre_counts.plot(kind='barh', figsize=(9, 5), color='steelblue', edgecolor='white')
plt.title('Top 15 Movie Genres')
plt.xlabel('Number of Movies')
plt.tight_layout()
plt.savefig('../results/genre_distribution.png', dpi=150, bbox_inches='tight')
plt.show()